# Hybrid Retrieval: Combining BM25 and Semantic Search

Combine keyword-based (BM25) and semantic (embedding) retrieval for better results.

## Output
- Verified hybrid retrieval working
- Comparison of BM25 vs Semantic vs Hybrid
- Ready for RAG pipeline in 07_rag_pipeline.ipynb

## Workflow
1. Check prerequisites (both indexes exist)
2. Load BM25 index
3. Load Semantic index
4. Create hybrid merge strategy
5. Test on sample queries
6. Compare all three methods

## 1. Setup and Prerequisites

Initializes paths and checks that both BM25 and Semantic indexes exist from phases 1-2.

**Output:** Project root and prerequisite file status

In [1]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

corpus_file = project_root / "data" / "processed" / "corpus.pkl"
bm25_index_file = project_root / "data" / "processed" / "bm25_index.pkl"
semantic_index_file = project_root / "data" / "processed" / "semantic_index" / "faiss_index"
books_data_file = project_root / "data" / "processed" / "books_sample.parquet"

print(f"Project root: {project_root}")
print()
print("Prerequisites check:")
print(f"  corpus.pkl exists: {corpus_file.exists()}")
print(f"  bm25_index.pkl exists: {bm25_index_file.exists()}")
print(f"  semantic_index/faiss_index exists: {semantic_index_file.exists()}")
print(f"  books_sample.parquet exists: {books_data_file.exists()}")

missing = [f for f in [corpus_file, bm25_index_file, semantic_index_file, books_data_file] if not f.exists()]

if missing:
    print(f"\nERROR: Missing files. Run 03_bm25 and 04_semantic notebooks first.")
else:
    print("\nAll prerequisites ready.")

Project root: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki

Prerequisites check:
  corpus.pkl exists: True
  bm25_index.pkl exists: True
  semantic_index/faiss_index exists: True
  books_sample.parquet exists: True

All prerequisites ready.


## 2. Load BM25 Index

Loads the BM25 keyword search index from phase 1.

**Output:** BM25 index confirmation

In [2]:
import pickle
import importlib

# Force reimport
if 'src.bm25' in sys.modules:
    importlib.reload(sys.modules['src.bm25'])

from src.bm25 import BM25Retriever

print("Loading BM25 index...")

bm25_retriever = BM25Retriever()
bm25_retriever.load(str(bm25_index_file))

with open(corpus_file, "rb") as f:
    corpus = pickle.load(f)

bm25_retriever.corpus = corpus

print(f"BM25 index loaded")
print(f"Corpus: {len(corpus):,} documents")

Loading BM25 index...
BM25 index loaded from /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl
BM25 index loaded
Corpus: 18,361 documents


## 3. Load Semantic Index

Loads the semantic embedding index from phase 2.

**Output:** Semantic index confirmation

In [3]:
# Force reimport
if 'src.semantic_retriever' in sys.modules:
    importlib.reload(sys.modules['src.semantic_retriever'])

from src.semantic_retriever import SemanticRetriever

print("Loading Semantic index...")

semantic_retriever = SemanticRetriever()
semantic_retriever.load(str(semantic_index_file))
semantic_retriever.corpus = corpus

print(f"Semantic index loaded")

Loading Semantic index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
Semantic index loaded


## 4. Create Hybrid Retrieval Function

Combines BM25 and Semantic results using simple merge: union of top-k results, deduplicated.

**Output:** Hybrid function ready to use

In [4]:
def hybrid_search(query: str, top_k: int = 5):
    """Hybrid search combining BM25 and semantic."""
    # Get results from both
    bm25_results = bm25_retriever.search(query, top_k)
    semantic_results = semantic_retriever.search(query, top_k)
    
    # Extract doc IDs
    bm25_ids = set(idx for idx, _ in bm25_results)
    semantic_ids = set(idx for idx, _ in semantic_results)
    
    # Union (combine) and limit to top_k
    combined_ids = list(bm25_ids | semantic_ids)[:top_k]
    
    return combined_ids

print("Hybrid retrieval function created")
print(f"Strategy: Union of BM25 and Semantic top-{5}")

Hybrid retrieval function created
Strategy: Union of BM25 and Semantic top-5


## 5. Test Hybrid Retrieval

Tests hybrid search on sample queries. Shows which results come from which method.

**Output:** Hybrid search results for each query

In [5]:
import pandas as pd

print("Testing Hybrid Retrieval")
print("=" * 80)

books_data = pd.read_parquet(books_data_file)

test_queries = ["python book", "mystery novel", "machine learning"]

for query in test_queries:
    hybrid_ids = hybrid_search(query, top_k=5)
    print(f"\nQuery: '{query}'")
    print(f"Results: {len(hybrid_ids)} documents")
    for rank, doc_id in enumerate(hybrid_ids, 1):
        book_title = books_data.iloc[doc_id]['product_title']
        rating = books_data.iloc[doc_id]['rating']
        print(f"  {rank}. Doc #{doc_id} - {book_title} ({rating}/5)")

Testing Hybrid Retrieval

Query: 'python book'
Results: 5 documents
  1. Doc #18030 - Introduction to Programming Using Python (Myprogramminglab) (2.0/5)
  2. Doc #15442 - Data Structures and Algorithms in Python (5.0/5)
  3. Doc #6771 - Python Cookbook (5.0/5)
  4. Doc #500 - Elements of Programming Interviews in Python: The Insiders' Guide (5.0/5)
  5. Doc #14452 - Python Basics: A Practical Introduction to Python 3 (5.0/5)

Query: 'mystery novel'
Results: 5 documents
  1. Doc #7488 - Classic Tales of Mystery (Leather-bound Classics) (5.0/5)
  2. Doc #5478 - Strange but True: A Novel (4.0/5)
  3. Doc #9353 - Mystery on Hidden Lane: An utterly gripping cozy mystery novel (An Eve Mallow Mystery Book 1) (4.0/5)
  4. Doc #9161 - All the Devils Are Here: A Novel (Chief Inspector Gamache Novel, 16) (5.0/5)
  5. Doc #11404 - The Family Upstairs: A Novel (5.0/5)

Query: 'machine learning'
Results: 5 documents
  1. Doc #7884 - Machine Learning (1.0/5)
  2. Doc #877 - Machine Learning: A Techn

## 6. Compare: BM25 vs Semantic vs Hybrid

Shows differences between methods on same query. Demonstrates why hybrid helps.

**Output:** Side-by-side comparison showing unique results from each method

In [6]:
print("Comparison: BM25 vs Semantic vs Hybrid")
print("=" * 80)

test_query = "good book recommendation"
print(f"\nQuery: '{test_query}'")

# Get results from each method
bm25_results = bm25_retriever.search(test_query, top_k=5)
semantic_results = semantic_retriever.search(test_query, top_k=5)
hybrid_ids = hybrid_search(test_query, top_k=5)

bm25_ids = set(idx for idx, _ in bm25_results)
semantic_ids = set(idx for idx, _ in semantic_results)

print(f"\nBM25 (keyword): {len(bm25_ids)} unique results")
for idx in list(bm25_ids)[:3]:
    print(f"  - {books_data.iloc[idx]['product_title'][:50]}")

print(f"\nSemantic (embedding): {len(semantic_ids)} unique results")
for idx in list(semantic_ids)[:3]:
    print(f"  - {books_data.iloc[idx]['product_title'][:50]}")

print(f"\nHybrid (combined): {len(hybrid_ids)} results")
print(f"  From BM25 only: {len(bm25_ids - semantic_ids)}")
print(f"  From Semantic only: {len(semantic_ids - bm25_ids)}")
print(f"  From both: {len(bm25_ids & semantic_ids)}")

print(f"\nHybrid advantage: Combines strengths of both methods")

Comparison: BM25 vs Semantic vs Hybrid

Query: 'good book recommendation'

BM25 (keyword): 5 unique results
  - The Terminal List: A Thriller (1)
  - Mrs. Everything: A Novel
  - The Lost Hours

Semantic (embedding): 5 unique results
  - The Internet For Dummies
  - Great Source Reader's Handbooks: Lesson Plan Book 
  - Later

Hybrid (combined): 5 results
  From BM25 only: 5
  From Semantic only: 5
  From both: 0

Hybrid advantage: Combines strengths of both methods


## 7. Verify Hybrid Works

Final verification that hybrid retrieval is working correctly and ready for RAG.

**Output:** Status and readiness for next phase

In [7]:
print("Hybrid Retrieval Verification")
print("=" * 80)

test_count = 3
all_working = True

for i in range(test_count):
    try:
        results = hybrid_search(f"test query {i}", top_k=5)
        if len(results) > 0 and len(results) <= 5:
            print(f"Test {i+1}: PASS (got {len(results)} results)")
        else:
            print(f"Test {i+1}: FAIL (invalid result count)")
            all_working = False
    except Exception as e:
        print(f"Test {i+1}: ERROR - {str(e)}")
        all_working = False

print()
if all_working:
    print("VERIFICATION PASSED: Hybrid retrieval ready")
    print("Next: Run 07_rag_pipeline.ipynb")
else:
    print("VERIFICATION FAILED: Check errors above")

Hybrid Retrieval Verification
Test 1: PASS (got 5 results)
Test 2: PASS (got 5 results)
Test 3: PASS (got 5 results)

VERIFICATION PASSED: Hybrid retrieval ready
Next: Run 07_rag_pipeline.ipynb
